In [21]:
import numpy as np
import torch
import os
import glob
import matplotlib.pyplot as plt

In [22]:
import pandas as pd

# Load the config file into a DataFrame
config_file_path = '/home/dm4/repo/spectral-norm-variance-regularization/bash_scripts/config/test.csv'
df = pd.read_csv(config_file_path)
# Strip leading and trailing spaces from column names
df.columns = df.columns.str.strip()
# Strip leading and trailing spaces from all entries
df = df.map(lambda x: x.strip().replace('\"','\'') if isinstance(x, str) else x)
# Iterate over each row in the DataFrame
for idx, row in df.iterrows():
    ckpt_path = f'/home/dm4/repo/spectral-norm-variance-regularization/ckpt/{row["dataset"]}_{row["usage"]}/{row["model"]}_{row["depth"]}/{row["lr"]}_{row["snr"]}_{row["snvr"]}'
    seeds = str(row["seed"]).split(';')

    # check mean and std of test accuracy already computed
    # accuracy is a string of the form 'mean ± std (n)' and n is the number of seeds
    # if n is same as the number of seeds, then skip this row
    if 'valid_acc' not in row:
        continue

    if '(' in row['valid_acc']:
        n = int(row['valid_acc'].split('(')[-1].split(')')[0])
        if n == len(seeds):
            continue

    valid_accs = []
    test_accs = []
    for seed in seeds:
        seed_path = os.path.join(ckpt_path, seed)
        best_epoch_path = os.path.join(seed_path, 'best_epoch.pth')
        if not os.path.exists(best_epoch_path):
            continue
        best_epoch = torch.load(best_epoch_path)
        valid_accs.append(best_epoch['valid_acc'])
        test_accs.append(best_epoch['test_acc'])        

    if not test_accs:
        df.at[idx, 'valid_acc'] = '-'
        df.at[idx, 'test_acc'] = '-'
    else:
        mean_acc = np.mean(valid_accs)
        std_acc = np.std(valid_accs)
        df.at[idx, 'valid_acc'] = f'{mean_acc:.2f} ± {std_acc:.2f} ({len(valid_accs)})'
        mean_acc = np.mean(test_accs)
        std_acc = np.std(test_accs)
        df.at[idx, 'test_acc'] = f'{mean_acc:.2f} ± {std_acc:.2f} ({len(test_accs)})'
    
df.to_csv(config_file_path, index=False)
df

,dataset,usage,model,depth,lr,snr,snvr,seed,valid_acc,test_acc,remark
0,cifar100,1.0,resnet,18,0.1,0.0,0.0,43;37;13;51;71;16;98;68;24;77,-,-,{}
1,cifar100,1.0,resnet,34,0.05,0.0,0.0,43;37;13;51;71,-,-,{}
2,cifar100,1.0,vgg,16,0.05,0.0,0.0,43;37;13;51;71,72.49 ± 0.42 (5),72.26 ± 0.55 (5),{}
3,cifar100,1.0,vgg,16,0.1,0.0,0.0,43;37;13;51;71,72.61 ± 0.41 (5),72.61 ± 0.28 (5),{}
4,cifar100,1.0,vgg,16,0.2,0.0,0.0,43;37;13;51;71,72.60 ± 0.25 (5),72.05 ± 0.23 (5),{}
5,cifar100,1.0,vgg,16,0.1,0.005,0.0,43;37;13;51;71,72.89 ± 0.41 (5),72.52 ± 0.37 (5),{}
6,cifar100,1.0,vgg_clf_4096,16,0.05,0.0,0.0,43;37;13;51;71,71.96 ± 0.18 (5),71.95 ± 0.17 (5),{'linear_dim':4096}
7,cifar100,1.0,vgg_clf_4096,16,0.1,0.0,0.0,43;37;13;51;71,72.42 ± 0.48 (5),72.21 ± 0.16 (5),{'linear_dim':4096}
8,cifar100,1.0,vgg_clf_4096,16,0.2,0.0,0.0,43;37;13;51;71,71.46 ± 0.53 (5),71.39 ± 0.15 (5),{'linear_dim':4096}
9,cifar100,1.0,vgg_clf_4096,16,0.1,0.1,0.0,43;37;13;51;71,70.90 ± 0.00 (1),70.48 ± 0.00 (1),{'linear_dim':4096}
